<a href="https://colab.research.google.com/github/melissa-04/tubitak-2209a-spatial-stemness-emt/blob/main/06_validation_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 06_validation_layers - Cell 1.1
# Setup, path validation, output directory

from google.colab import drive
drive.mount('/content/drive')

import os, sys

BASE    = '/content/drive/MyDrive/Tubitak-2209a'
RESULTS = f'{BASE}/04_results'
FIGURES = f'{BASE}/05_figures'

# Create output directories (no-op if they already exist)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)
print(f'Results dir: {RESULTS}')
print(f'Figures dir: {FIGURES}\n')

# Verify the five input files required by this notebook
INPUTS = {
    'regions'  : f'{BASE}/01_processed_data/combined_with_regions.csv',
    'scrna'    : f'{BASE}/02_scRNA_analysis/scrna_filtered.h5ad',
    'cytotrace': f'{BASE}/02_scRNA_analysis/cytotrace2_epithelial_results.csv',
    'allscores': f'{BASE}/03_spatial_analysis/spatial_all_scores.csv',
    'visium'   : f'{BASE}/03_spatial_analysis/visium_6patients_raw.h5ad',
}

print('--- INPUT FILES ---')
missing = []
for name, path in INPUTS.items():
    if os.path.exists(path):
        print(f'  OK      {name:<10} {os.path.getsize(path)/1e6:8.1f} MB   {os.path.basename(path)}')
    else:
        print(f'  MISSING {name:<10} -> {path}')
        missing.append(name)

print('\nAll inputs present.' if not missing else f'\nSTOP - missing: {missing}')

# Record library versions for reproducibility
print('\n--- LIBRARIES ---')
try:
    import scanpy as sc, anndata
except ImportError:
    print('  installing scanpy (1-2 min)...')
    !pip install -q scanpy
    import scanpy as sc, anndata

try:
    import statsmodels
except ImportError:
    print('  installing statsmodels...')
    !pip install -q statsmodels
    import statsmodels

import pandas as pd, numpy as np, scipy

print(f'  python      {sys.version.split()[0]}')
print(f'  scanpy      {sc.__version__}')
print(f'  anndata     {anndata.__version__}')
print(f'  pandas      {pd.__version__}')
print(f'  numpy       {np.__version__}')
print(f'  scipy       {scipy.__version__}')
print(f'  statsmodels {statsmodels.__version__}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results dir: /content/drive/MyDrive/Tubitak-2209a/04_results
Figures dir: /content/drive/MyDrive/Tubitak-2209a/05_figures

--- INPUT FILES ---
  OK      regions         1.9 MB   combined_with_regions.csv
  OK      scrna         358.6 MB   scrna_filtered.h5ad
  OK      cytotrace       3.8 MB   cytotrace2_epithelial_results.csv
  OK      allscores       4.3 MB   spatial_all_scores.csv
  OK      visium        144.0 MB   visium_6patients_raw.h5ad

All inputs present.

--- LIBRARIES ---
  python      3.13.15
  scanpy      1.12.4
  anndata     0.13.4
  pandas      2.3.3
  numpy       2.1.3
  scipy       1.16.3
  statsmodels 0.15.0


/tmp/ipykernel_7400/1549470735.py:58: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f'  scanpy      {sc.__version__}')
/tmp/ipykernel_7400/1549470735.py:59: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f'  anndata     {anndata.__version__}')


In [3]:
!pip install -q "pandas==2.3.3"
print('\nDone. Now: Runtime > Restart session, then re-run Cell 1.1.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 61.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.

Done. Now: Runtime > Restart session, then re-run Cell 1.1.


In [2]:
# 06_validation_layers - Cell 1.2
# Load data and verify marker gene availability
import scanpy as sc, pandas as pd, numpy as np
sc.settings.verbosity = 0

# Load spatial object (raw counts + region labels)
vis = sc.read_h5ad(INPUTS['visium'])
print(f'Visium      : {vis.shape[0]} spots x {vis.shape[1]} genes')
print(f'  obs cols  : {list(vis.obs.columns)}')
print(f'  X is raw  : max={vis.X.max():.0f}, integer={float(vis.X.max()) == int(vis.X.max())}')

# Load single-cell object and CytoTRACE 2 scores
adata = sc.read_h5ad(INPUTS['scrna'])
cyto  = pd.read_csv(INPUTS['cytotrace'], index_col=0)
print(f'\nscRNA       : {adata.shape[0]} cells x {adata.shape[1]} genes')
print(f'CytoTRACE 2 : {cyto.shape[0]} cells, cols={list(cyto.columns)}')

# Load spot-level scores (stemness + EMT + composition)
scores = pd.read_csv(INPUTS['allscores'])
print(f'\nScores      : {scores.shape[0]} rows x {scores.shape[1]} cols')
print(f'  stemness defined in {scores.spatial_stemness.notna().sum()} spots')

# Marker panels for the three validation layers
MARKERS = {
    'Identity'  : ['CD44', 'CD24', 'ALDH1A1', 'SOX2', 'NANOG', 'POU5F1'],
    'Activity'  : ['MKI67', 'PCNA'],
    'Behavior'  : ['VIM', 'CDH1'],
}

print('\n--- MARKER AVAILABILITY ---')
print(f"{'layer':<10} {'gene':<9} {'visium':<8} {'scrna':<8} {'spot_detect%':<13} {'cell_detect%'}")
avail = {}
for layer, genes in MARKERS.items():
    for g in genes:
        in_v = g in vis.var_names
        in_s = g in adata.var_names
        # fraction of spots / cells with non-zero counts
        pv = float((vis[:, g].X > 0).sum()) / vis.n_obs * 100 if in_v else np.nan
        ps = float((adata[:, g].X > 0).sum()) / adata.n_obs * 100 if in_s else np.nan
        avail[g] = (layer, in_v, in_s, pv, ps)
        print(f'{layer:<10} {g:<9} {str(in_v):<8} {str(in_s):<8} '
              f'{pv:>8.1f}     {ps:>8.1f}')

usable = [g for g, v in avail.items() if v[1] and v[2]]
dropped = [g for g, v in avail.items() if not (v[1] and v[2])]
print(f'\nUsable in both datasets : {len(usable)}/{len(avail)} -> {usable}')
if dropped:
    print(f'Not usable              : {dropped}')

Visium      : 15611 spots x 28925 genes
  obs cols  : ['patientid', 'subtype', 'Classification', 'region_class', 'exclude_reason', 'array_row', 'array_col', 'pxl_row', 'pxl_col', 'analyze', 'n_nontumor_2ring', 'n_nontumor_3ring', 'n_valid_3ring', 'tumor_region']
  X is raw  : max=7336, integer=True

scRNA       : 100064 cells x 29733 genes
CytoTRACE 2 : 28844 cells, cols=['CytoTRACE2_Score', 'CytoTRACE2_Potency', 'CytoTRACE2_Relative', 'preKNN_CytoTRACE2_Score', 'preKNN_CytoTRACE2_Potency', 'celltype_major']

Scores      : 15611 rows x 22 cols
  stemness defined in 10263 spots

--- MARKER AVAILABILITY ---
layer      gene      visium   scrna    spot_detect%  cell_detect%
Identity   CD44      True     True         77.5         52.8
Identity   CD24      True     True         96.6         32.8
Identity   ALDH1A1   True     True          9.9          4.4
Identity   SOX2      True     True          0.5          0.1
Identity   NANOG     True     True          0.0          0.1
Identity   POU5F

In [4]:
# 06_validation_layers - Cell 1.2b
# Extend marker panel, apply detection threshold, finalize panels
CANDIDATES = {
    'Identity': ['CD44', 'CD24', 'ALDH1A1', 'ALDH1A3', 'PROM1', 'ITGA6',
                 'SOX2', 'NANOG', 'POU5F1'],
    'Activity': ['MKI67', 'PCNA', 'TOP2A'],
    'Behavior': ['VIM', 'CDH1', 'ZEB1', 'SNAI2'],
}

DETECT_MIN = 5.0  # minimum % of spots with non-zero counts

rows = []
for layer, genes in CANDIDATES.items():
    for g in genes:
        in_v, in_s = g in vis.var_names, g in adata.var_names
        pv = float((vis[:, g].X > 0).sum()) / vis.n_obs * 100 if in_v else 0.0
        ps = float((adata[:, g].X > 0).sum()) / adata.n_obs * 100 if in_s else 0.0
        rows.append({'layer': layer, 'gene': g, 'in_visium': in_v, 'in_scrna': in_s,
                     'spot_detect_pct': round(pv, 2), 'cell_detect_pct': round(ps, 2),
                     'usable': in_v and in_s and pv >= DETECT_MIN})

marker_qc = pd.DataFrame(rows)
print(marker_qc.to_string(index=False))

# Final panels: only markers passing the detection threshold
PANELS = {L: marker_qc.query('layer == @L and usable').gene.tolist()
          for L in CANDIDATES}
EXCLUDED = marker_qc.query('not usable')[['layer', 'gene', 'spot_detect_pct']]

print(f'\n--- FINAL PANELS (detection >= {DETECT_MIN}% of spots) ---')
for L, g in PANELS.items():
    print(f'  {L:<9}: {g}')
print(f'\n--- EXCLUDED (reported as negative result) ---')
print(EXCLUDED.to_string(index=False) if len(EXCLUDED) else '  none')

# Composite CD44-CD24 score (continuous proxy for the Al-Hajj CD44+/CD24- phenotype)
CD44_CD24 = all(g in vis.var_names for g in ['CD44', 'CD24'])
print(f'\nCD44/CD24 composite available: {CD44_CD24}')

marker_qc.to_csv(f'{RESULTS}/marker_qc.csv', index=False)
print(f'Saved -> {RESULTS}/marker_qc.csv')

   layer    gene  in_visium  in_scrna  spot_detect_pct  cell_detect_pct  usable
Identity    CD44       True      True            77.54            52.84    True
Identity    CD24       True      True            96.62            32.82    True
Identity ALDH1A1       True      True             9.86             4.37    True
Identity ALDH1A3       True      True            14.21             4.21    True
Identity   PROM1       True      True            37.72             4.64    True
Identity   ITGA6       True      True            33.14            13.90    True
Identity    SOX2       True      True             0.47             0.11   False
Identity   NANOG       True      True             0.02             0.13   False
Identity  POU5F1       True      True             1.09             0.50   False
Activity   MKI67       True      True            27.18             4.93    True
Activity    PCNA       True      True            62.73            19.64    True
Activity   TOP2A       True      True   

In [5]:
# 06_validation_layers - Cell 1.3
# Spot-level validation: identity, activity, behavior vs spatial_stemness
from scipy.stats import spearmanr, wilcoxon

# Log-normalize a copy, keeping raw counts untouched (same recipe as Stage 5)
vn = vis.copy()
sc.pp.normalize_total(vn, target_sum=1e4)
sc.pp.log1p(vn)
print(f'Normalized: max {vn.X.max():.3f} (was {vis.X.max():.0f})')

# Module score per layer
for layer, genes in PANELS.items():
    sc.tl.score_genes(vn, gene_list=genes, score_name=f'score_{layer}',
                      ctrl_size=50, random_state=42, use_raw=False)

# Composite CD44-CD24: continuous proxy for the CD44+/CD24- stem phenotype
cd44 = np.asarray(vn[:, 'CD44'].X.todense()).ravel()
cd24 = np.asarray(vn[:, 'CD24'].X.todense()).ravel()
z = lambda a: (a - a.mean()) / a.std()
vn.obs['CD44_minus_CD24'] = z(cd44) - z(cd24)

# Assemble per-spot table: individual genes + module scores + stemness
ALL_GENES = [g for gs in PANELS.values() for g in gs]
expr = pd.DataFrame(np.asarray(vn[:, ALL_GENES].X.todense()),
                    index=vn.obs_names, columns=ALL_GENES)
expr['CD44_minus_CD24'] = vn.obs['CD44_minus_CD24'].values
for layer in PANELS:
    expr[f'score_{layer}'] = vn.obs[f'score_{layer}'].values
expr['patientid'] = vn.obs['patientid'].values

stem = scores.set_index('SpotID')['spatial_stemness']
expr['spatial_stemness'] = stem.reindex(expr.index).values
d = expr[expr.spatial_stemness.notna()].copy()
print(f'Spots with stemness: {len(d)}')

# Within-patient Spearman, then patient-level Wilcoxon
LAYER_OF = {g: L for L, gs in PANELS.items() for g in gs}
LAYER_OF.update({'CD44_minus_CD24': 'Identity',
                 **{f'score_{L}': L for L in PANELS}})
PATIENTS = sorted(d.patientid.unique())

rows = []
for var, layer in LAYER_OF.items():
    rs = [spearmanr(s.spatial_stemness, s[var])[0]
          for _, s in d.groupby('patientid', observed=True)]
    try:
        wp = wilcoxon(rs)[1]
    except Exception:
        wp = np.nan
    rows.append({'layer': layer, 'variable': var, 'level': 'spot',
                 'median_r': np.median(rs), 'n_positive': int(sum(r > 0 for r in rs)),
                 'n_patients': len(rs), 'wilcoxon_p': wp,
                 **{f'r_{p}': r for p, r in zip(PATIENTS, rs)}})

spot_val = pd.DataFrame(rows)
cols = ['layer', 'variable', 'median_r', 'n_positive', 'wilcoxon_p']
print('\n--- SPOT-LEVEL VALIDATION ---')
for L in ['Identity', 'Activity', 'Behavior']:
    print(f'\n{L}:')
    sub = spot_val[spot_val.layer == L][cols].copy()
    sub[['median_r', 'wilcoxon_p']] = sub[['median_r', 'wilcoxon_p']].round(3)
    print(sub.to_string(index=False))

spot_val.to_csv(f'{RESULTS}/validation_spot_level.csv', index=False)
print(f'\nSaved -> {RESULTS}/validation_spot_level.csv')

Normalized: max 8.105 (was 7336)
Spots with stemness: 10263

--- SPOT-LEVEL VALIDATION ---

Identity:
   layer        variable  median_r  n_positive  wilcoxon_p
Identity            CD44    -0.017           1       0.312
Identity            CD24    -0.170           2       0.156
Identity         ALDH1A1     0.016           5       0.156
Identity         ALDH1A3     0.005           3       0.688
Identity           PROM1    -0.034           2       0.844
Identity           ITGA6    -0.053           2       0.156
Identity CD44_minus_CD24     0.039           4       0.562
Identity  score_Identity    -0.065           1       0.312

Activity:
   layer       variable  median_r  n_positive  wilcoxon_p
Activity          MKI67    -0.003           3       1.000
Activity           PCNA    -0.023           1       0.219
Activity          TOP2A     0.002           4       0.844
Activity score_Activity     0.018           4       0.562

Behavior:
   layer       variable  median_r  n_positive  wilcoxon

In [6]:
from scipy.stats import spearmanr, wilcoxon, mannwhitneyu, kruskal

epi = adata[adata.obs.celltype_major.isin(['Cancer Epithelial', 'Normal Epithelial'])].copy()
epi.obs['CT2'] = cyto.reindex(epi.obs_names)['CytoTRACE2_Score'].values
print(f'Epithelial cells: {epi.shape[0]}, with CytoTRACE score: {epi.obs.CT2.notna().sum()}')

sc.pp.normalize_total(epi, target_sum=1e4)
sc.pp.log1p(epi)

for layer, genes in PANELS.items():
    present = [g for g in genes if g in epi.var_names]
    sc.tl.score_genes(epi, gene_list=present, score_name=f'score_{layer}',
                      ctrl_size=50, random_state=42, use_raw=False)

c44 = np.asarray(epi[:, 'CD44'].X.todense()).ravel()
c24 = np.asarray(epi[:, 'CD24'].X.todense()).ravel()
epi.obs['CD44_minus_CD24'] = z(c44) - z(c24)

ALL_G = [g for gs in PANELS.values() for g in gs]
cdf = pd.DataFrame(np.asarray(epi[:, ALL_G].X.todense()), index=epi.obs_names, columns=ALL_G)
cdf['CD44_minus_CD24'] = epi.obs['CD44_minus_CD24'].values
for L in PANELS:
    cdf[f'score_{L}'] = epi.obs[f'score_{L}'].values
cdf['CT2']       = epi.obs['CT2'].values
cdf['patient']   = epi.obs['orig.ident'].values
cdf['celltype']  = epi.obs['celltype_major'].values
cdf['subtype']   = epi.obs['subtype'].values
cdf = cdf.dropna(subset=['CT2'])

# Part A: within-patient correlations, all epithelium vs cancer epithelium only
def per_patient_corr(frame, label, min_cells=100):
    out = []
    for var, layer in LAYER_OF.items():
        rs = [spearmanr(g.CT2, g[var])[0]
              for _, g in frame.groupby('patient', observed=True) if len(g) >= min_cells]
        rs = [r for r in rs if not np.isnan(r)]
        try:
            wp = wilcoxon(rs)[1]
        except Exception:
            wp = np.nan
        out.append({'layer': layer, 'variable': var, 'level': label,
                    'median_r': np.median(rs), 'n_positive': int(sum(r > 0 for r in rs)),
                    'n_patients': len(rs), 'wilcoxon_p': wp})
    return pd.DataFrame(out)

cell_all    = per_patient_corr(cdf, 'cell_all_epithelium')
cell_cancer = per_patient_corr(cdf[cdf.celltype == 'Cancer Epithelial'], 'cell_cancer_only')

print('\n--- CELL-LEVEL, ALL EPITHELIUM ---')
for L in ['Identity', 'Activity', 'Behavior']:
    sub = cell_all[cell_all.layer == L][['variable', 'median_r', 'n_positive', 'n_patients', 'wilcoxon_p']]
    print(f'\n{L}:'); print(sub.round(3).to_string(index=False))

print('\n--- CELL-LEVEL, CANCER EPITHELIUM ONLY ---')
sub = cell_cancer[cell_cancer.layer == 'Identity'][['variable', 'median_r', 'n_positive', 'n_patients', 'wilcoxon_p']]
print(sub.round(3).to_string(index=False))

# Part B: hierarchy validation - the biology-based identity evidence
print('\n--- HIERARCHY VALIDATION ---')
nor = cdf[cdf.celltype == 'Normal Epithelial'].CT2
can = cdf[cdf.celltype == 'Cancer Epithelial'].CT2
u, p = mannwhitneyu(nor, can, alternative='greater')
cliffs = 2 * u / (len(nor) * len(can)) - 1
print(f'Normal ({nor.median():.3f}, n={len(nor)}) vs Cancer ({can.median():.3f}, n={len(can)})')
print(f'  Mann-Whitney U p={p:.2e}, Cliff\'s delta={cliffs:+.3f}')

canc = cdf[cdf.celltype == 'Cancer Epithelial']
print('\nStemness by subtype (Cancer Epithelial):')
groups = []
for s, g in canc.groupby('subtype', observed=True):
    print(f'  {s:<10} n={len(g):6d}  median={g.CT2.median():.3f}  mean={g.CT2.mean():.3f}')
    groups.append(g.CT2.values)
print(f'  Kruskal-Wallis p={kruskal(*groups)[1]:.2e}')

val_cell = pd.concat([cell_all, cell_cancer], ignore_index=True)
val_cell.to_csv(f'{RESULTS}/validation_cell_level.csv', index=False)
print(f'\nSaved -> {RESULTS}/validation_cell_level.csv')

Epithelial cells: 28844, with CytoTRACE score: 28844


/tmp/ipykernel_7400/2211126646.py:34: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs = [spearmanr(g.CT2, g[var])[0]



--- CELL-LEVEL, ALL EPITHELIUM ---

Identity:
       variable  median_r  n_positive  n_patients  wilcoxon_p
           CD44     0.047          13          22       0.113
           CD24     0.020          12          22       0.799
        ALDH1A1     0.015          12          18       0.167
        ALDH1A3     0.068          18          20       0.000
          PROM1     0.063          12          16       0.002
          ITGA6     0.094          17          22       0.000
CD44_minus_CD24     0.039          12          22       0.147
 score_Identity     0.003          11          22       0.726

Activity:
      variable  median_r  n_positive  n_patients  wilcoxon_p
         MKI67     0.175          20          21         0.0
          PCNA     0.162          18          22         0.0
         TOP2A     0.198          21          22         0.0
score_Activity     0.177          20          22         0.0

Behavior:
      variable  median_r  n_positive  n_patients  wilcoxon_p
       

In [7]:
from scipy.stats import spearmanr, wilcoxon

S_GENES = ['MCM5','PCNA','TYMS','FEN1','MCM2','MCM4','RRM1','UNG','GINS2','MCM6',
 'CDCA7','DTL','PRIM1','UHRF1','MLF1IP','HELLS','RFC2','RPA2','NASP','RAD51AP1',
 'GMNN','WDR76','SLBP','CCNE2','UBR7','POLD3','MSH2','ATAD2','RAD51','RRM2',
 'CDC45','CDC6','EXO1','TIPIN','DSCC1','BLM','CASP8AP2','USP1','CLSPN','POLA1',
 'CHAF1B','BRIP1','E2F8']
G2M_GENES = ['HMGB2','CDK1','NUSAP1','UBE2C','BIRC5','TPX2','TOP2A','NDC80','CKS2',
 'NUF2','CKS1B','MKI67','TMPO','CENPF','TACC3','FAM64A','SMC4','CCNB2','CKAP2L',
 'CKAP2','AURKB','BUB1','KIF11','ANP32E','TUBB4B','GTSE1','KIF20B','HJURP','CDCA3',
 'HN1','CDC20','TTK','CDC25C','KIF2C','RANGAP1','NCAPD2','DLGAP5','CDCA2','CDCA8',
 'ECT2','KIF23','HMMR','AURKA','PSRC1','ANLN','LBR','CKAP5','CENPE','CTCF','NEK2',
 'G2E3','GAS2L3','CBX5','CENPA']

s_cell   = [g for g in S_GENES   if g in epi.var_names]
g2m_cell = [g for g in G2M_GENES if g in epi.var_names]
print(f'Cell cycle genes found (cells): S {len(s_cell)}/{len(S_GENES)}, G2M {len(g2m_cell)}/{len(G2M_GENES)}')

sc.tl.score_genes_cell_cycle(epi, s_genes=s_cell, g2m_genes=g2m_cell)
cdf['S_score']   = epi.obs.loc[cdf.index, 'S_score'].values
cdf['G2M_score'] = epi.obs.loc[cdf.index, 'G2M_score'].values
cdf['nFeature']  = epi.obs.loc[cdf.index, 'nFeature_RNA'].values
cdf['nCount']    = epi.obs.loc[cdf.index, 'nCount_RNA'].values
cdf['CT2_preKNN'] = cyto.reindex(cdf.index)['preKNN_CytoTRACE2_Score'].values

print('\nCell cycle phase distribution:')
print(epi.obs.phase.value_counts().to_string())

def diag(frame, target, group_col, label, min_n=100):
    out = []
    for var in ['nFeature', 'nCount', 'S_score', 'G2M_score', 'score_Activity']:
        if var not in frame.columns:
            continue
        rs = [spearmanr(g[target], g[var])[0]
              for _, g in frame.groupby(group_col, observed=True) if len(g) >= min_n]
        rs = [r for r in rs if not np.isnan(r)]
        try:
            wp = wilcoxon(rs)[1]
        except Exception:
            wp = np.nan
        out.append({'level': label, 'target': target, 'predictor': var,
                    'median_r': round(np.median(rs), 3),
                    'n_positive': int(sum(r > 0 for r in rs)),
                    'n_groups': len(rs), 'wilcoxon_p': wp})
    return pd.DataFrame(out)

print('\n--- CELL LEVEL: what drives CytoTRACE2_Score? ---')
d_cell = diag(cdf, 'CT2', 'patient', 'cell')
print(d_cell.to_string(index=False))

print('\n--- CELL LEVEL: same for pre-KNN score (before smoothing) ---')
d_pre = diag(cdf.dropna(subset=['CT2_preKNN']), 'CT2_preKNN', 'patient', 'cell_preKNN')
print(d_pre.to_string(index=False))

print('\n--- SPOT LEVEL: what drives spatial_stemness? ---')
s_spot   = [g for g in S_GENES   if g in vn.var_names]
g2m_spot = [g for g in G2M_GENES if g in vn.var_names]
sc.tl.score_genes_cell_cycle(vn, s_genes=s_spot, g2m_genes=g2m_spot)

reg = pd.read_csv(INPUTS['regions'])
reg['SpotID'] = reg.patientid.astype(str) + '_' + reg.barcode.astype(str)
reg = reg.set_index('SpotID')

d['S_score']   = vn.obs.loc[d.index, 'S_score'].values
d['G2M_score'] = vn.obs.loc[d.index, 'G2M_score'].values
d['nFeature']  = reg.reindex(d.index)['nFeature_RNA'].values
d['nCount']    = reg.reindex(d.index)['nCount_RNA'].values

d_spot = diag(d, 'spatial_stemness', 'patientid', 'spot', min_n=50)
print(d_spot.to_string(index=False))

diagnostics = pd.concat([d_cell, d_pre, d_spot], ignore_index=True)
diagnostics.to_csv(f'{RESULTS}/cytotrace_diagnostics.csv', index=False)
print(f'\nSaved -> {RESULTS}/cytotrace_diagnostics.csv')

Cell cycle genes found (cells): S 42/43, G2M 54/54

Cell cycle phase distribution:
phase
G1     19508
S       6781
G2M     2555

--- CELL LEVEL: what drives CytoTRACE2_Score? ---
level target      predictor  median_r  n_positive  n_groups  wilcoxon_p
 cell    CT2       nFeature     0.256          18        22    0.002851
 cell    CT2         nCount     0.233          18        22    0.000941
 cell    CT2        S_score     0.127          20        22    0.000042
 cell    CT2      G2M_score     0.141          20        22    0.000005
 cell    CT2 score_Activity     0.177          20        22    0.000002

--- CELL LEVEL: same for pre-KNN score (before smoothing) ---
      level     target      predictor  median_r  n_positive  n_groups  wilcoxon_p
cell_preKNN CT2_preKNN       nFeature     0.193          18        22    0.002195
cell_preKNN CT2_preKNN         nCount     0.172          19        22    0.000305
cell_preKNN CT2_preKNN        S_score     0.115          20        22    0.00008

In [9]:
from scipy.stats import rankdata, mannwhitneyu, kruskal

sc_idx = scores.set_index('SpotID')
for c in ['pEMT_puram', 'EMT_hallmark', 'EMT_tan', 'Cancer Epithelial', 'CAFs']:
    d[c] = sc_idx.reindex(d.index)[c].values
print(f'Spot table ready: {d.shape[0]} spots, EMT + composition columns merged')

CC = ['S_score', 'G2M_score', 'nFeature']

def partial_spearman(frame, x, y, covars):
    if frame[x].nunique() < 3 or frame[y].nunique() < 3:
        return np.nan
    A = np.column_stack([rankdata(frame[c]) for c in covars])
    A = np.c_[np.ones(len(A)), A]
    resid = lambda a: a - A @ np.linalg.lstsq(A, a, rcond=None)[0]
    return spearmanr(resid(rankdata(frame[x])), resid(rankdata(frame[y])))[0]

def resid_on(frame, target, covars):
    A = np.column_stack([rankdata(frame[c]) for c in covars])
    A = np.c_[np.ones(len(A)), A]
    a = frame[target].values
    return a - A @ np.linalg.lstsq(A, a, rcond=None)[0]

print('\n=== A) HIERARCHY VALIDATION UNDER CELL-CYCLE CONTROL ===')
cdf['CT2_resid'] = resid_on(cdf, 'CT2', CC)
for tgt, lbl in [('CT2', 'raw'), ('CT2_resid', 'cycle-controlled')]:
    nor = cdf[cdf.celltype == 'Normal Epithelial'][tgt]
    can = cdf[cdf.celltype == 'Cancer Epithelial'][tgt]
    u, p = mannwhitneyu(nor, can, alternative='greater')
    print(f'  Normal vs Cancer [{lbl:17s}] Cliff delta={2*u/(len(nor)*len(can))-1:+.3f}, p={p:.2e}')

canc = cdf[cdf.celltype == 'Cancer Epithelial'].copy()
canc['CT2_resid'] = resid_on(canc, 'CT2', CC)
for tgt, lbl in [('CT2', 'raw'), ('CT2_resid', 'cycle-controlled')]:
    meds = canc.groupby('subtype', observed=True)[tgt].median()
    kp = kruskal(*[g[tgt].values for _, g in canc.groupby('subtype', observed=True)])[1]
    print(f'  Subtype     [{lbl:17s}] ' +
          ' < '.join(f'{s}({v:+.3f})' for s, v in meds.sort_values().items()) +
          f', Kruskal p={kp:.2e}')

print('\n=== B) IDENTITY LAYER UNDER CELL-CYCLE CONTROL (cancer epithelium) ===')
print('    (raw and partial computed on the SAME patient set for fair comparison)')
rows = []
for var in PANELS['Identity'] + ['CD44_minus_CD24']:
    raw, par = [], []
    for _, g in canc.groupby('patient', observed=True):
        if len(g) < 100 or g[var].nunique() < 3:
            continue
        r = spearmanr(g.CT2, g[var])[0]
        pr = partial_spearman(g, 'CT2', var, CC)
        if np.isnan(r) or np.isnan(pr):
            continue
        raw.append(r); par.append(pr)
    n = len(raw)
    rows.append({'variable': var, 'n_patients': n,
                 'raw_r': round(np.median(raw), 3), 'raw_pos': f'{sum(r>0 for r in raw)}/{n}',
                 'raw_p': round(wilcoxon(raw)[1], 4),
                 'partial_r': round(np.median(par), 3), 'partial_pos': f'{sum(r>0 for r in par)}/{n}',
                 'partial_p': round(wilcoxon(par)[1], 4)})
identity_cc = pd.DataFrame(rows)
print(identity_cc.to_string(index=False))

print('\n=== C) MAIN FINDING: stemness x EMT under increasing control (spot level) ===')
rows = []
for sig in ['pEMT_puram', 'EMT_hallmark', 'EMT_tan']:
    raw  = [spearmanr(g.spatial_stemness, g[sig])[0] for _, g in d.groupby('patientid', observed=True)]
    cyc  = [partial_spearman(g, 'spatial_stemness', sig, CC) for _, g in d.groupby('patientid', observed=True)]
    full = [partial_spearman(g, 'spatial_stemness', sig, CC + ['Cancer Epithelial', 'CAFs'])
            for _, g in d.groupby('patientid', observed=True)]
    rows.append({'signature': sig,
                 'raw_r': round(np.median(raw), 3),  'raw_pos': f'{sum(r>0 for r in raw)}/6',
                 'raw_p': round(wilcoxon(raw)[1], 4),
                 'cycle_r': round(np.median(cyc), 3), 'cycle_pos': f'{sum(r>0 for r in cyc)}/6',
                 'cycle_p': round(wilcoxon(cyc)[1], 4),
                 'full_r': round(np.median(full), 3), 'full_pos': f'{sum(r>0 for r in full)}/6',
                 'full_p': round(wilcoxon(full)[1], 4)})
main_cc = pd.DataFrame(rows)
print(main_cc.to_string(index=False))
print('\n  raw   = no covariates')
print('  cycle = S_score + G2M_score + nFeature')
print('  full  = cycle + Cancer Epithelial fraction + CAF fraction')

identity_cc.to_csv(f'{RESULTS}/cellcycle_controlled_identity.csv', index=False)
main_cc.to_csv(f'{RESULTS}/cellcycle_controlled_main.csv', index=False)
print(f'\nSaved -> {RESULTS}/cellcycle_controlled_identity.csv')
print(f'Saved -> {RESULTS}/cellcycle_controlled_main.csv')

Spot table ready: 10263 spots, EMT + composition columns merged

=== A) HIERARCHY VALIDATION UNDER CELL-CYCLE CONTROL ===
  Normal vs Cancer [raw              ] Cliff delta=+0.449, p=0.00e+00
  Normal vs Cancer [cycle-controlled ] Cliff delta=+0.491, p=0.00e+00
  Subtype     [raw              ] ER+(+0.049) < HER2+(+0.126) < TNBC(+0.196), Kruskal p=0.00e+00
  Subtype     [cycle-controlled ] ER+(-0.056) < HER2+(-0.007) < TNBC(+0.047), Kruskal p=0.00e+00

=== B) IDENTITY LAYER UNDER CELL-CYCLE CONTROL (cancer epithelium) ===
    (raw and partial computed on the SAME patient set for fair comparison)
       variable  n_patients  raw_r raw_pos  raw_p  partial_r partial_pos  partial_p
           CD44          20  0.076   15/20 0.0027      0.010       11/20     0.7012
           CD24          20  0.004   10/20 0.5706     -0.039        8/20     0.1769
        ALDH1A1          13  0.021   11/13 0.0081     -0.018        3/13     0.0681
        ALDH1A3          16  0.030   14/16 0.0034     -0.012 